# Fact payment_gateway_logs
1. read the data from the silver payment_gateway_logs
2. join the silver payment_gateway_logs table, silver transactions table, gold dim_accounts table and gold dim_date table
3. select the required columns
- Degenearate dimensions: txn_id, gateway_name, gateway_status, response_code, device_type, geo_location
- Foreign keys: account_sk, customer_sk, processed_date_key
- Measures: processing_time_ms

In [0]:
#Imports
from pyspark.sql.functions import col

In [0]:
payment_gateway_logs_df = spark.read.table("neo_bank.bronze.payment_gateway_logs")
transactions_df = spark.read.table("neo_bank.bronze.transactions")
accounts_df = spark.read.table("neo_bank.gold.dim_accounts")
date_df = spark.read.table("neo_bank.gold.dim_date")

In [0]:
payment_gateway_logs_df = payment_gateway_logs_df.select(
    col("txn_id"),
    col("gateway_name"),
    col("gateway_status"),
    col("response_code").cast("int"),
    col("processing_time_ms").cast("int"),
    col("device_type"),
    col("geo_location"),
    col("processed_timestamp").cast("date").alias("processed_date")
)


In [0]:
fact_payment_gateway_logs_df = (
    payment_gateway_logs_df.alias("pgl")
    .join(
        transactions_df.alias("t"),
        col("pgl.txn_id")==col("t.txn_id")
    )
    .join(
        accounts_df.alias("a"),
        col("t.account_id")==col("a.account_id"),
        "left"
    )
    .join(
        date_df.alias("d"),
        col("pgl.processed_date")==col("d.full_date"),
        "left"
    )

)

In [0]:
fact_payment_gateway_logs_df = fact_payment_gateway_logs_df.select(
    col("pgl.txn_id"),
    col("pgl.gateway_name"),
    col("pgl.gateway_status"),
    col("pgl.response_code"),
    col("pgl.device_type"),
    col("pgl.geo_location"),
    col("a.account_sk"),
    col("a.customer_sk"),
    col("d.date_key").alias("processed_date_key"),
    col("pgl.processing_time_ms")
)


In [0]:
(
    fact_payment_gateway_logs_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("neo_bank.gold.fact_payment_gateway_logs")
)

In [0]:
%sql
select * from neo_bank.gold.fact_payment_gateway_logs